# Act 4 — RAG & Agent Evaluation

**Goal:** Show that RAG and agent pipelines need *different* eval metrics from plain Q&A.

| Pattern | What changes | New metrics needed |
|---|---|---|
| Plain Q&A | LLM answers from memory | Relevance, safety, helpfulness |
| **RAG** | LLM answers from retrieved context | Faithfulness, groundedness, context relevance |
| **Agent** | LLM decides which tool to call | Tool correctness, task completion |

No vector DB required — we use a hardcoded restaurant knowledge base.

In [1]:
import os, re, json
import mlflow
from openai import OpenAI
from dotenv import load_dotenv
from mlflow.genai.scorers import scorer

load_dotenv()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Session4 / Act 4 - RAG & Agent Eval")
mlflow.openai.autolog()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY", "ollama"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def _chat(messages: list, max_tokens: int = 150) -> str:
    return client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0,
    ).choices[0].message.content

print(f"Model: {MODEL}")

2026/04/10 12:53:14 INFO mlflow.tracking.fluent: Experiment with name 'Session4 / Act 4 - RAG & Agent Eval' does not exist. Creating a new experiment.


Model: llama3.2:1b


---
## Knowledge base (simulated retrieval)

In production this would be a vector database. Here we use a Python dict  
with keyword-based retrieval — same eval logic, zero infrastructure.

In [2]:
KNOWLEDGE_BASE = {
    "behrouz biryani": (
        "Behrouz Biryani is a premium delivery-only brand specializing in Mughlai-style "
        "dum biryani. Founded in 2015, it operates across Mumbai, Delhi, Bengaluru, and Pune. "
        "Signature dishes include the Mughlai Chicken Dum Biryani and the Gosht Dum Biryani. "
        "They do not have dine-in outlets. Average delivery time is 40–60 minutes."
    ),
    "truffles bengaluru": (
        "Truffles is a casual dining restaurant chain in Bengaluru known for burgers, "
        "steaks, and comfort food. Popular outlets are in Koramangala, Indiranagar, and HSR Layout. "
        "Open daily from 12:00 PM to 11:00 PM. Average cost for two is Rs.700–900. "
        "Famous for their signature Truffles Burger and BBQ Chicken."
    ),
    "saravana bhavan": (
        "Saravana Bhavan is a South Indian vegetarian restaurant chain with outlets across "
        "Chennai, Mumbai, Delhi, and internationally. Known for its affordable thali meals, "
        "filter coffee, and dosas. A typical thali costs Rs.150–250. "
        "Open from 7:00 AM to 11:00 PM. No alcohol served."
    ),
    "indian accent delhi": (
        "Indian Accent is a fine dining restaurant in New Delhi known for modern Indian cuisine. "
        "Located in The Lodhi hotel, Lodhi Road. Chef Manish Mehrotra is the culinary director. "
        "Tasting menu starts at Rs.4500 per person. Reservations required. "
        "Awarded multiple times in Asia's 50 Best Restaurants."
    ),
}

def retrieve(question: str) -> str:
    """Keyword-based retrieval from the knowledge base."""
    q = question.lower()
    for key, doc in KNOWLEDGE_BASE.items():
        if any(word in q for word in key.split()):
            return doc
    return "No specific information found in the knowledge base for this query."

# Test retrieval
context = retrieve("Tell me about Truffles in Bengaluru")
print("Retrieved:", context[:100], "...")

Retrieved: Truffles is a casual dining restaurant chain in Bengaluru known for burgers, steaks, and comfort foo ...


---
# Part A — RAG Evaluation

```
query → retrieve(query) → context → LLM(query + context) → answer
```

The key question: **did the LLM stay within the context, or did it hallucinate?**

In [3]:
def rag_bot(question: str, context: str = "") -> str:
    """Generate an answer grounded in the provided context."""
    if not context:
        context = retrieve(question)
    prompt = (
        f"Use ONLY the following information to answer the question. "
        f"Do not add facts not mentioned below.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return _chat([{"role": "user", "content": prompt}])

# Preview
ans = rag_bot("What biryani does Behrouz serve?")
print("RAG answer:", ans)

RAG answer: Behrouz Biryani serves Mughlai-style dum biryani, specifically the Mughlai Chicken Dum Biryani and the Gosht Dum Biryani.


Trace(trace_id=tr-e4333a7211481b0d5bf88b6341a84a64)

In [4]:
rag_eval_data = [
    {
        "inputs": {
            "question": "What kind of biryani does Behrouz serve?",
            "context": KNOWLEDGE_BASE["behrouz biryani"],
            "expected_response": "Mughlai dum biryani",
        }
    },
    {
        "inputs": {
            "question": "What are Truffles' opening hours in Bengaluru?",
            "context": KNOWLEDGE_BASE["truffles bengaluru"],
            "expected_response": "12:00 PM to 11:00 PM daily",
        }
    },
    {
        "inputs": {
            "question": "Does Saravana Bhavan serve alcohol?",
            "context": KNOWLEDGE_BASE["saravana bhavan"],
            "expected_response": "No, Saravana Bhavan does not serve alcohol",
        }
    },
    {
        "inputs": {
            "question": "How much does Indian Accent tasting menu cost?",
            "context": KNOWLEDGE_BASE["indian accent delhi"],
            "expected_response": "Tasting menu starts at Rs.4500 per person",
        }
    },
]

In [5]:
def ask_judge(prompt: str) -> dict:
    judge_client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY", "ollama"),
        base_url=os.getenv("OPENAI_BASE_URL"),
    )
    raw = judge_client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        max_tokens=100, temperature=0,
    ).choices[0].message.content.strip()
    for m in re.finditer(r'\{[^{}]+\}', raw, re.DOTALL):
        try: return json.loads(m.group())
        except json.JSONDecodeError: continue
    return {"result": "unknown", "reason": raw}


@scorer
def context_relevance(inputs: dict, outputs: str) -> bool:
    """Heuristic: does the retrieved context contain keywords from the question?"""
    context = inputs.get("context", "")
    question_words = set(inputs.get("question", "").lower().split()) - {
        "what", "is", "the", "a", "an", "does", "do", "how", "much", "are"
    }
    context_words = set(context.lower().split())
    overlap = len(question_words & context_words)
    return overlap >= 2  # at least 2 meaningful shared words


@scorer
def faithfulness(inputs: dict, outputs: str) -> float:
    """LLM judge: does the answer stay within the provided context (no hallucination)?"""
    context = inputs.get("context", "")
    verdict = ask_judge(
        f"Context: {context[:500]}\n\nAnswer: {outputs}\n\n"
        "Rate how faithful this answer is to the context (1=completely fabricated, 5=fully grounded). "
        'Reply ONLY with JSON: {"score": <1-5>, "reason": "one sentence"}'
    )
    try:
        return (float(verdict.get("score", 3)) - 1) / 4
    except (TypeError, ValueError):
        return 0.5


@scorer
def answer_groundedness(inputs: dict, outputs: str) -> bool:
    """LLM judge: does the answer avoid adding facts not in context (no hallucination)?"""
    context = inputs.get("context", "")
    verdict = ask_judge(
        f"Context: {context[:500]}\n\nAnswer: {outputs}\n\n"
        "Does the answer contain ONLY information that is explicitly stated in the context above? "
        'Reply ONLY with JSON: {"result": "yes" or "no", "reason": "one sentence"}'
    )
    return verdict.get("result", "no").lower() == "yes"


print("RAG scorers: context_relevance, faithfulness, answer_groundedness")

RAG scorers: context_relevance, faithfulness, answer_groundedness


In [6]:
rag_results = mlflow.genai.evaluate(
    data=rag_eval_data,
    predict_fn=lambda question, context="", expected_response="": rag_bot(question, context),
    scorers=[context_relevance, faithfulness, answer_groundedness],
)

print("RAG eval scores:")
for k, v in sorted(rag_results.metrics.items()):
    print(f"  {k}: {float(v):.2f}")

2026/04/10 12:53:16 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/10 12:53:16 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?]

RAG eval scores:
  answer_groundedness/mean: 0.25
  context_relevance/mean: 0.75
  faithfulness/mean: 0.25


In [7]:
import pandas as pd
pd.set_option("display.max_colwidth", 55)
df_rag = rag_results.tables["eval_results"]
cols = [c for c in df_rag.columns if "/value" in c or c == "outputs"]
df_rag[cols]

,context_relevance/value,faithfulness/value,answer_groundedness/value
0,True,0.25,True
1,False,0.25,False
2,True,0.25,False
3,True,0.25,False


---
# Part B — Agent Evaluation

An agent decides *which tool to call* based on the query.  
Eval must check the **process** (did it pick the right tool?), not just the final output.

**Tools available to ZomatoBot:**
- `search_restaurant(name)` — look up a specific restaurant  
- `suggest_alternatives(cuisine, city)` — recommend alternatives

In [8]:
def search_restaurant(name: str) -> str:
    """Tool: look up restaurant facts from the knowledge base."""
    name_lower = name.lower()
    for key, doc in KNOWLEDGE_BASE.items():
        if any(word in name_lower for word in key.split()):
            return doc
    return f"No information found for '{name}'."

def suggest_alternatives(cuisine: str, city: str) -> str:
    """Tool: suggest alternatives based on cuisine and city."""
    suggestions = [doc[:80] for key, doc in KNOWLEDGE_BASE.items()
                   if city.lower() in key or cuisine.lower() in doc.lower()]
    return "\n".join(suggestions[:2]) if suggestions else "No alternatives found."

TOOLS = {"search_restaurant": search_restaurant, "suggest_alternatives": suggest_alternatives}
print("Tools registered:", list(TOOLS.keys()))

Tools registered: ['search_restaurant', 'suggest_alternatives']


In [9]:
AGENT_PROMPT = """You are ZomatoBot with access to these tools:
- search_restaurant(name): Look up details about a specific restaurant
- suggest_alternatives(cuisine, city): Get alternative restaurant suggestions

If you need a tool, respond EXACTLY with:
TOOL: tool_name(argument)

Otherwise respond with:
ANSWER: your response

User query: {query}"""

def zomato_agent(query: str, expected_tool: str = "") -> str:
    """Simple ReAct-style agent. expected_tool is ignored (used by scorer via inputs)."""
    # Step 1: decide what to do
    decision = _chat(
        [{"role": "user", "content": AGENT_PROMPT.format(query=query)}],
        max_tokens=60,
    )

    # Step 2: if tool call, execute and follow up
    tool_match = re.match(r'TOOL:\s*(\w+)\((.*)\)', decision.strip())
    if tool_match:
        tool_name = tool_match.group(1)
        tool_arg  = tool_match.group(2).strip().strip('"\'')
        tool_fn   = TOOLS.get(tool_name)
        tool_result = tool_fn(tool_arg) if tool_fn else f"Unknown tool: {tool_name}"

        # Step 3: generate final answer with tool output
        final = _chat([
            {"role": "user",      "content": query},
            {"role": "assistant", "content": decision},
            {"role": "user",      "content": f"Tool result: {tool_result[:300]}\n\nNow answer the user."},
        ])
        return f"[used {tool_name}] {final}"

    # No tool needed — direct answer
    return decision.replace("ANSWER:", "").strip()

# Preview
print(zomato_agent("Tell me about Behrouz Biryani"))

TOOL: zomato search restaurant name "Behrouz Biryani"


Trace(trace_id=tr-79b5b1443ec73630acb584be063044c8)

In [10]:
agent_eval_data = [
    {
        "inputs": {
            "query": "What biryani does Behrouz serve?",
            "expected_tool": "search_restaurant",
        }
    },
    {
        "inputs": {
            "query": "Suggest alternatives to Behrouz in Mumbai",
            "expected_tool": "suggest_alternatives",
        }
    },
    {
        "inputs": {
            "query": "What are Truffles' opening hours?",
            "expected_tool": "search_restaurant",
        }
    },
    {
        "inputs": {
            "query": "Hi, how are you?",
            "expected_tool": "",   # no tool needed
        }
    },
]

@scorer
def correct_tool_called(inputs: dict, outputs: str) -> bool:
    """Did the agent call the expected tool? (True if no tool expected and none called)"""
    expected = inputs.get("expected_tool", "")
    if not expected:
        return "[used " not in outputs   # no tool should have been called
    return expected in outputs

@scorer
def task_completed(inputs: dict, outputs: str) -> bool:
    """LLM judge: did the agent actually answer the user's question?"""
    q = inputs.get("query", "")
    verdict = ask_judge(
        f"User asked: {q}\nAgent responded: {outputs}\n\n"
        'Did the agent answer the user\'s question? '
        'Reply ONLY with JSON: {"result": "yes" or "no", "reason": "one sentence"}'
    )
    return verdict.get("result", "no").lower() == "yes"


agent_results = mlflow.genai.evaluate(
    data=agent_eval_data,
    predict_fn=lambda query, expected_tool="": zomato_agent(query, expected_tool),
    scorers=[correct_tool_called, task_completed],
)

print("Agent eval scores:")
for k, v in sorted(agent_results.metrics.items()):
    print(f"  {k}: {float(v):.2f}")

2026/04/10 12:53:23 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/4 [Elapsed: 00:00, Remaining: ?]

2026/04/10 12:53:31 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'correct_tool_called': 1/4 failed. Check individual trace assessments for detailed error messages.


Agent eval scores:
  correct_tool_called/mean: 0.67
  task_completed/mean: 0.50


In [11]:
df_agent = agent_results.tables["eval_results"]
cols = [c for c in df_agent.columns if "/value" in c or c == "outputs"]
df_agent[cols]

,correct_tool_called/value,task_completed/value
0,True,False
1,None,False
2,False,True
3,True,True


---
## Key takeaways

**RAG eval requires:**
- `context_relevance` — was the right context retrieved?
- `faithfulness` — did the LLM stick to the context (no hallucination)?
- `answer_groundedness` — did it add facts beyond the context?

**Agent eval requires:**
- `correct_tool_called` — did it pick the right tool for the job?
- `task_completed` — did the final answer actually help the user?

**The big insight:** For agents, a correct final answer with the *wrong tool path* is still a problem  
— it signals the routing logic is fragile. Eval must inspect the process, not just the output.

➡️ Next: [05_querying_mlflow.ipynb](05_querying_mlflow.ipynb)